In [1]:
import os
os.environ['LOGS_DIRECTORY'] = '../eval_logs'

import sys
sys.path.insert(0, '..')

In [2]:
import json
import random

from tqdm.auto import tqdm
from pydantic import BaseModel
from pydantic_ai import Agent

from ingest import read_repo_data, chunk_documents
from main import initialize_index, initialize_agent
from logs import log_interaction_to_file, LOG_DIR

In [3]:
REPO_OWNER = "huggingface"
REPO_NAME = "pytorch-image-models"

In [4]:
docs = read_repo_data(REPO_OWNER, REPO_NAME)
pytorch_img_models_chunks = chunk_documents(docs, 2000, 1000)
index, vindex = initialize_index()
agent = initialize_agent(index, vindex)

Starting AI FAQ Assistant for huggingface/pytorch-image-models
Initializing data ingestion...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Data indexing completed successfully!
Initializing search agent...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Agent initialized successfully!


In [8]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about the PyTorch Image Models (timm) repository.

Based on the provided repository content (documentation, model descriptions, and code snippets), generate realistic questions that users (developers, ML engineers, or researchers) might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.

IMPORTANT:
- Questions should be answerable using the provided content
- Do NOT invent information outside the repository
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model='gpt-4o-mini',
    output_type=QuestionsList
)

In [9]:
sample = random.sample(pytorch_img_models_chunks, 5)
prompt_docs = [
    {
        "title": d.get("title", ""),
        "content": d.get("chunk", ""),
        "source": d.get("filename", "")
    }
    for d in sample
]
prompt = f"""
Generate questions based on the following repository content:

{json.dumps(prompt_docs, indent=2)}
"""

result = await question_generator.run(prompt)
questions = result.output.questions

In [10]:
for q in tqdm(questions):
    print(q)

    result = await agent.run(user_prompt=q)
    print(result.output)

    log_interaction_to_file(
        agent,
        result.new_messages(),
        source='ai-generated'
    )

    print()

  0%|          | 0/15 [00:00<?, ?it/s]

What are the main architectural components of ResNet models in the timm repository?
I couldn't find specific details on the architectural components of ResNet models in the timm repository. However, I can provide you with a general overview of ResNet architecture.

ResNet (Residual Networks) typically consists of the following main architectural components:

1. **Convolutional Layers:** These are the fundamental building blocks of the network where the convolution operation is applied.

2. **Batch Normalization:** This layer normalizes the output of the previous layer to improve training speed and stability.

3. **Activation Functions:** Usually, ReLU (Rectified Linear Unit) is used after each convolution operation to introduce non-linearity.

4. **Residual Connections:** The hallmark of ResNet is the addition of skip connections that allow the input to be added to the output of a series of convolutional layers. This helps in mitigating the vanishing gradient problem and allows for the

Traceback (most recent call last):
  File "/Users/jishnuvsarmah/Documents/aihero/app/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3745, in run_code
    await eval(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/z1/s30qpky14s9dlr028scsdqc00000gn/T/ipykernel_96405/1342262676.py", line 4, in <module>
    result = await agent.run(user_prompt=q)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jishnuvsarmah/Documents/aihero/app/.venv/lib/python3.13/site-packages/pydantic_ai/agent/abstract.py", line 281, in run
    async with self.iter(
               ~~~~~~~~~^
        user_prompt=user_prompt,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
        spec=spec,
        ^^^^^^^^^^
    ) as agent_run:
    ^
  File "/Users/jishnuvsarmah/.local/share/uv/python/cpython-3.13.5-macos-aarch64-none/lib/python3.13/contextlib.py", line 235, in __aexit__
    await self.gen.athrow(value)
  File "/Users/jishnuvsarmah/Documents/aihero/app/.venv